### Real data

In [1]:
import numpy as np
import scipy.io
import torch
from torch_geometric.data import Data
from pathlib import Path
from torch_geometric.datasets import KarateClub, WikipediaNetwork, Planetoid
from prettytable import PrettyTable
from itertools import chain
import networkx as nx
import random
import torch.nn as nn
from sklearn.metrics import adjusted_rand_score
import random
from models import GEE, GNN
from time import time
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
def _dedup_edges(edge: torch.Tensor) -> torch.Tensor:
    """
    Converts a directed edge list to an undirected one and removes duplicate edges.
    This is a helper function to ensure graph consistency.

    Args:
        edge (torch.Tensor): A [2, num_edges] tensor representing the edge index.

    Returns:
        torch.Tensor: A processed edge index tensor with no duplicates or self-loops.
    """
    row = torch.minimum(edge[0], edge[1])
    col = torch.maximum(edge[0], edge[1])
    edge_undirected = torch.stack([row, col], dim=0)   

    return torch.unique(edge_undirected, dim=1)

def load_real_data(name: str, root: str = "../datasets"):
    """
    Loads real-world graph datasets from various sources.
    It supports three loading mechanisms:
      1. PyTorch Geometric's online datasets (e.g., "Cora").
      2. Local .mat files (e.g., "your_dataset.mat").
      3. A directory of .npy files for adj, features, and labels.

    Args:
        name (str): The name of the dataset.
        root (str, optional): The root directory for datasets. Defaults to "./datasets".

    Returns:
        torch_geometric.data.Data: A PyG Data object containing graph info.
    """
    # --- 1. Load from PyG's online library ---
    online_map = {
        "KarateClub":        lambda: KarateClub(),
        "Chameleon":         lambda: WikipediaNetwork(root, name="Chameleon"),
        "Cora":              lambda: Planetoid(root, name="Cora"),
        "Citeseer":          lambda: Planetoid(root, name="Citeseer"),
    }
    if name in online_map:
        ds = online_map[name]()       
        data  = ds[0]
        data.edge_index = _dedup_edges(data.edge_index)  
        data.k = ds.num_classes 
        return data

    # --- 2 & 3. Load from local files ---
    root = Path(root)
    mat_file = root / f"{name}.mat"
    npy_dir  = root / name

    if mat_file.exists():                                  # -------- mat --------
        mat  = scipy.io.loadmat(mat_file)
        edge = torch.tensor(mat["Edge"][:, :2].T, dtype=torch.long) - 1
        edge = _dedup_edges(edge)
        if edge.max() == 1:
            edge -= 1

        lbl_key = "Label" if "Label" in mat else "Y"
        y = torch.tensor(mat[lbl_key].ravel(), dtype=torch.long)
        data = Data(x=None, edge_index=edge, y=y)

    elif npy_dir.is_dir():                                 # -------- npy --------
        adj   = np.load(npy_dir / f"{name}_adj.npy")
        feat  = np.load(npy_dir / f"{name}_feat.npy").astype(np.float32)
        label = np.load(npy_dir / f"{name}_label.npy")

        row, col = np.where(adj > 0)
        edge = torch.tensor(np.vstack([row, col]), dtype=torch.long)
        if np.allclose(adj, adj.T):
            edge = _dedup_edges(edge)
        # if edge.min() == 1:
        #     edge -= 1

        data = Data(x=torch.tensor(feat), edge_index=edge,
                    y=torch.tensor(label, dtype=torch.long))
    else:
        raise FileNotFoundError(f"Dataset '{name}' not found in {root}.")

    data.k = len(torch.unique(data.y))
        
    return data

def write(file_path, content, dataset):
    """
    Appends experiment results for a given run to a tab-separated file.

    Args:
        file_path (str): The path to the output file.
        content (dict): A dictionary mapping method names to their results.
        dataset (str): The dataset we used.
    """
    with open(file_path, "a") as f:
        for method, arr in content.items():
            arr = np.asarray(arr, dtype=float)
            # flat = arr.flatten()
            ari_str = ",".join(f"{x}" for x in arr)
            line = f"{dataset}\t{method}\t{ari_str}\n"
            f.write(line)

In [3]:
def run_real_data(dataname, return_embdict=False):
    """
    Runs the GEE, GNN, and hybrid models (GG and GG-C) on a specified dataset
    and prints their performance (ARI and runtime).

    Args:
        dataname (str): The name of the dataset to run on.
        return_embdict (bool, optional): Whether to return the learned embeddings. Defaults to False.

    Returns:
        tuple: A tuple containing performance metrics and learned embeddings.
    """
    data = load_real_data(dataname)
    edge_list = [(int(data.edge_index[0, i]), int(data.edge_index[1, i]), 1) for i in range(data.edge_index.shape[1])]
    labels = data.y.numpy()
    n = data.y.size(0) 

    # --- Baseline 1: GEE ---
    st = time()
    model_gee = GEE.UnSup_Gee(edge_list, n, data.k)
    Z_GEE, yhat = model_gee.GEE_unsup() 
    et = time()
    t_GEE = et - st
    ari_GEE = adjusted_rand_score(labels, yhat)
    print(f"GEE running time: {t_GEE:.4f}s, ARI: {ari_GEE:.4f}")

    # --- Baseline 2: GNN (initialized randomly) ---
    initializer = nn.init.xavier_uniform_
    z = initializer(torch.empty(n, data.k))
    data.x = z  
    st = time()
    ari_GNN, Z_GNN, Z_GNN_dict = GNN.gnn_clustering_with_dmon(data, labels=labels, return_embdict=return_embdict)
    et = time()
    t_GNN = et - st
    print(f"GNN running time: {t_GNN:.4f}s, ARI: {ari_GNN:.4f}")

    # --- Our Proposed Model: GEE-powered GNN (GG) ---
    z = torch.tensor(Z_GEE, dtype=torch.float)
    data.x = z  
    st = time()
    ari_GG, Z_GG, Z_GG_dict = GNN.gnn_clustering_with_dmon(data, labels=labels, return_embdict=return_embdict)
    et = time()
    t_GG = et - st
    print(f"GG running time: {t_GG:.4f}s, ARI: {ari_GG:.4f}")


    ari_t = ([ari_GEE, ari_GNN, ari_GG], [t_GEE, t_GNN, t_GG])
    Z_dict = Z_GNN_dict, Z_GG_dict
    return ari_t, Z_dict

In [4]:
# --- 1. Define the list of datasets to be analyzed ---
# Datasets are categorized by their source format (.npy, .mat, or integrated).
data_npy_list = ["ACM","BAT","DBLP","EAT","UAT","Wiki"]
data_mat_list = ["email", "Gene", "IIP", "lastfm", "polblogs", "TerroristRel"]
data_int_list = ["KarateClub", "Chameleon", "Cora", "Citeseer"]
data_list = data_npy_list +  data_mat_list + data_int_list


# --- 2. Iterate through datasets to collect statistics ---
# We will store statistics in a dictionary for easy table generation.
rows = {
    "num_nodes" : [],
    "num_edges" : [],
    "num_classes": [],
    "feat_dim"  : [],
}
for name in data_list:
    data = load_real_data(name) 
    print(data)          
    rows["num_nodes" ].append(data.y.shape[0])
    rows["num_edges" ].append(data.edge_index.size(1))
    rows["num_classes"].append(data.k)
    rows["feat_dim"  ].append(None if data.x is None else data.x.size(1))


# --- 3. Format and print the statistics using PrettyTable ---
table_long = PrettyTable()
table_long.field_names = ["Info"] + data_list
for info, vals in rows.items():
    table_long.add_row([info] + vals)

print(table_long)

Data(x=[3025, 1870], edge_index=[2, 13128], y=[3025], k=3)
Data(x=[131, 81], edge_index=[2, 1074], y=[131], k=4)
Data(x=[4057, 334], edge_index=[2, 3528], y=[4057], k=4)
Data(x=[399, 203], edge_index=[2, 5995], y=[399], k=4)
Data(x=[1190, 239], edge_index=[2, 13599], y=[1190], k=4)
Data(x=[2405, 4973], edge_index=[2, 16523], y=[2405], k=17)
Data(edge_index=[2, 16706], y=[1005], k=42)
Data(edge_index=[2, 1672], y=[1103], k=2)
Data(edge_index=[2, 630], y=[219], k=3)
Data(edge_index=[2, 27806], y=[7624], k=18)
Data(edge_index=[2, 16715], y=[1224], k=2)
Data(edge_index=[2, 8592], y=[881], k=3)
Data(x=[34, 34], edge_index=[2, 78], y=[34], train_mask=[34], k=4)
Data(x=[2277, 2325], edge_index=[2, 31421], y=[2277], train_mask=[2277, 10], val_mask=[2277, 10], test_mask=[2277, 10], k=5)
Data(x=[2708, 1433], edge_index=[2, 5278], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], k=7)
Data(x=[3327, 3703], edge_index=[2, 4552], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=

In [ ]:
# =============================================================================
# Main script for conducting replicated evaluations on real-world datasets.
#
# This script systematically evaluates the performance of different models
# (GEE, GNN, GG) on real datasets. For each dataset, it
# executes multiple independent replications (N runs), each with a unique
# random seed, to ensure robust evaluation of ARI and execution time.
# =============================================================================
import json
import random
torch.manual_seed(901)
random.seed(901)
np.random.seed(901)
for dataname in data_list:
    N = 50
    ari_all = {'GEE':[], 'GNN':[], "GG":[]}
    time_all = {'GEE':[], 'GNN':[], "GG":[]}

    # Run simulations and collect results
    for r in range(N):
        torch.manual_seed(r)
        random.seed(r)
        np.random.seed(r)
        ari_t, _ = run_real_data(dataname)
        ari_list, time_list= ari_t
        for i, m in enumerate(ari_all):
            ari_all[m].append(ari_list[i])
            time_all[m].append(time_list[i])
    
    write(f"results/real_data/ari.txt", ari_all, dataname)
    write(f"results/real_data/time.txt", time_all, dataname)
